# J48 Decision Tree (C4.5)

J48 is an implementation of the C4.5 decision tree algorithm, which is an extension of the ID3 algorithm. It's one of the most popular decision tree algorithms used in machine learning.

## Key Concepts:

- **Information Gain**: Uses information gain (entropy-based) for splitting
- **Pruning**: Includes post-pruning to avoid overfitting
- **Continuous Attributes**: Can handle both categorical and continuous attributes
- **Missing Values**: Can handle missing values in the dataset
- **Rule Generation**: Can generate classification rules from the tree

## When to Use:

- When you need interpretable models
- When dealing with mixed data types (categorical and continuous)
- When you have missing values
- When you want both classification and rule extraction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_classification
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Dataset 1: Iris Dataset

We'll use the classic Iris dataset to demonstrate J48 (C4.5) decision tree.

In [ ]:
# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Feature shape: {X.shape}")
print(f"Classes: {target_names}")
print(f"Features: {feature_names}")
print(f"\nClass distribution: {np.bincount(y)}")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

## Train J48 (C4.5) Decision Tree

Note: scikit-learn's DecisionTreeClassifier with 'entropy' criterion implements C4.5-like behavior.

In [ ]:
# Initialize Decision Tree with entropy criterion (C4.5-like)
j48 = DecisionTreeClassifier(
    criterion='entropy',  # Information gain (C4.5 uses entropy)
    max_depth=None,       # No depth limit initially
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

# Train the model
j48.fit(X_train, y_train)

# Make predictions
y_pred = j48.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(f"Tree depth: {j48.get_depth()}")
print(f"Number of leaves: {j48.get_n_leaves()}")

## Visualize the Decision Tree

In [ ]:
# Plot the decision tree
plt.figure(figsize=(20, 10))
plot_tree(j48, 
          feature_names=feature_names,
          class_names=target_names,
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('J48 (C4.5) Decision Tree - Iris Dataset', fontsize=16)
plt.tight_layout()
plt.show()

## Model Evaluation

In [ ]:
# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names,
            yticklabels=target_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - J48 Decision Tree')
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
# Get feature importance
feature_importance = j48.feature_importances_

# Create DataFrame for visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("Feature Importance:")
print(importance_df)

# Plot feature importance
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='Importance', y='Feature')
plt.title('Feature Importance - J48 Decision Tree')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Pruning the Tree

C4.5 includes pruning to avoid overfitting. We can control this with max_depth and min_samples parameters.

In [ ]:
# Train pruned tree
j48_pruned = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=3,  # Limit depth to prevent overfitting
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

j48_pruned.fit(X_train, y_train)
y_pred_pruned = j48_pruned.predict(X_test)
accuracy_pruned = accuracy_score(y_test, y_pred_pruned)

print(f"Original tree accuracy: {accuracy:.4f}")
print(f"Pruned tree accuracy: {accuracy_pruned:.4f}")
print(f"Original tree depth: {j48.get_depth()}")
print(f"Pruned tree depth: {j48_pruned.get_depth()}")

In [ ]:
# Visualize pruned tree
plt.figure(figsize=(15, 8))
plot_tree(j48_pruned, 
          feature_names=feature_names,
          class_names=target_names,
          filled=True,
          rounded=True,
          fontsize=12)
plt.title('Pruned J48 Decision Tree (max_depth=3)', fontsize=16)
plt.tight_layout()
plt.show()

## Dataset 2: Synthetic Classification Dataset

In [ ]:
# Create synthetic dataset
X_syn, y_syn = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_classes=3,
    n_clusters_per_class=2,
    random_state=42
)

print(f"Synthetic dataset shape: {X_syn.shape}")
print(f"Class distribution: {np.bincount(y_syn)}")

In [ ]:
# Split and train
X_train_syn, X_test_syn, y_train_syn, y_test_syn = train_test_split(
    X_syn, y_syn, test_size=0.3, random_state=42, stratify=y_syn
)

j48_syn = DecisionTreeClassifier(criterion='entropy', random_state=42)
j48_syn.fit(X_train_syn, y_train_syn)
y_pred_syn = j48_syn.predict(X_test_syn)
accuracy_syn = accuracy_score(y_test_syn, y_pred_syn)

print(f"Accuracy on synthetic dataset: {accuracy_syn:.4f}")
print(f"Tree depth: {j48_syn.get_depth()}")
print(f"Number of leaves: {j48_syn.get_n_leaves()}")

## Cross-Validation

In [ ]:
# Perform cross-validation
cv_scores = cross_val_score(j48, X_train, y_train, cv=5)

print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## Extract Rules from Tree

C4.5 can generate classification rules. Let's extract rules from our tree.

In [ ]:
def extract_rules(tree, feature_names, class_names):
    """Extract classification rules from decision tree"""
    tree_ = tree.tree_
    feature_name = [
        feature_names[i] if i != -2 else "undefined!"
        for i in tree_.feature
    ]

    paths = []
    path = []
    
    def recurse(node, path, direction):
        if tree_.feature[node] != -2:
            name = feature_name[node]
            threshold = tree_.threshold[node]
            
            # Left child
            p_left = path + [f"{name} <= {threshold:.2f}"]
            recurse(tree_.children_left[node], p_left, "left")
            
            # Right child
            p_right = path + [f"{name} > {threshold:.2f}"]
            recurse(tree_.children_right[node], p_right, "right")
        else:
            class_idx = np.argmax(tree_.value[node])
            class_name = class_names[class_idx]
            rule = " AND ".join(path)
            paths.append((rule, class_name))
    
    recurse(0, [], "root")
    return paths

# Extract rules from pruned tree
rules = extract_rules(j48_pruned, feature_names, target_names)

print("Classification Rules from Pruned Tree:")
print("=" * 60)
for i, (rule, class_name) in enumerate(rules, 1):
    print(f"\nRule {i}:")
    print(f"IF {rule}")
    print(f"THEN class = {class_name}")

## Predict on New Data

In [ ]:
# Function to predict new samples
def predict_iris(sample):
    prediction = j48.predict([sample])[0]
    probabilities = j48.predict_proba([sample])[0]
    
    class_name = target_names[prediction]
    return class_name, probabilities

# Test with new samples
new_samples = [
    [5.1, 3.5, 1.4, 0.2],  # Likely setosa
    [6.3, 2.8, 5.1, 1.5],  # Likely versicolor
    [6.4, 3.2, 5.3, 2.3]   # Likely virginica
]

for sample in new_samples:
    class_name, probs = predict_iris(sample)
    print(f"\nSample: {sample}")
    print(f"Predicted: {class_name}")
    print("Probabilities:")
    for name, prob in zip(target_names, probs):
        print(f"  {name}: {prob:.4f}")

## Summary

### Key Takeaways:

1. **Information Gain**: Uses entropy-based information gain for splitting (C4.5 approach)
2. **Pruning**: Can prune trees to prevent overfitting
3. **Mixed Data**: Handles both categorical and continuous attributes
4. **Rule Extraction**: Can generate interpretable classification rules
5. **Missing Values**: Can handle missing values in the dataset

### Advantages:
- Highly interpretable and explainable
- Handles both categorical and continuous data
- Can generate classification rules
- Handles missing values
- No need for feature scaling
- Non-parametric (no assumptions about data distribution)

### Limitations:
- Can overfit without proper pruning
- Can be unstable (small changes in data can lead to different trees)
- May create biased trees if some classes dominate
- Decision boundaries are axis-aligned (rectangular)
- May not perform well on linearly separable data compared to other models